In [ ]:
# ar/python-101/hard/02-exploring-corpus
# Generated companion notebook for the PyDA course.
# Run cells top-to-bottom (or in any order) to follow the lesson.

print("PyDA — ready 🚀")


In [ ]:
# 💾 Load the course datasets into this environment
# The course data files live in the PyDA repo; pull them so
# `open("…")` / `pd.read_csv("…")` work exactly like on disk.
import os
def _fetch(name, aliases=()):
    if os.path.exists(name):
        return
    url = f"https://raw.githubusercontent.com/abderrahim-lectures/python-data-analysis-course/main/public/datasets/{name}"
    os.system(f"curl -sL -o {name} {url}")
    for alias in aliases:
        if not os.path.exists(alias):
            os.system(f"cp {name} {alias}")

_fetch("slm-corpus.csv", ())


استكشف قبل أن تعالج

تحميل البيانات هو الخطوة الأولى. الخطوة الثانية هي فهم ما حمّلته. قد يحتوي المتن على قيم مفقودة، أو صفوف مكررة، أو محارف مشفّرة تبدو كالبيانات التالفة، أو نصوصًا قصيرة جدًا لتكون مفيدة. خمس دقائق من الاستكشاف الآن توفر ساعات من التصحيح لاحقًا.

## المفاهيم الأساسية

### عدّ الصفوف والأعمدة

أبسط الإحصائيات تخبرك كثيرًا. متن من 5 صفوف لن يُنتج نموذجًا مفيدًا؛ متن من 50,000 صف قد يحتاج إلى تحميل مجزّأ:


In [ ]:
import csv

with open("slm-corpus.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Rows:    {len(rows)}")
print(f"Columns: {list(rows[0].keys())}")


### قياس طول النص

تحتاج النماذج اللغوية إلى نص كافٍ لتعلم الأنماط. افحص إجمالي عدد المحارف ومتوسط طول الصف:


In [ ]:
total_chars = sum(len(row["text"]) for row in rows)
avg_len = total_chars / len(rows) if rows else 0

print(f"Total characters: {total_chars:,}")
print(f"Average row length: {avg_len:.0f} characters")


متن بمتوسط 10 محارف في الصف قصير جدًا — لن يكون لدى النموذج سياق كافٍ لتعلم متتاليات الكلمات.

### معاينة نص عيّنة

اقرأ بضعة صفوف لتكوّن فكرة عن المحتوى. ما اللغة التي يكتب بها؟ ما الموضوعات التي يغطيها؟ هل النص نظيف أم مشوّش؟


In [ ]:
for i, row in enumerate(rows[:5]):
    preview = row["text"][:150].replace("\n", " ")
    print(f"[{i}] {preview}...")


### اكتشاف التكرارات

تضخّم الصفوف المكررة أعداد الكلمات دون إضافة معلومات جديدة. اكتشفها بتحويل الصفوف إلى مجموعة:


In [ ]:
unique_texts = set(row["text"] for row in rows)
print(f"Unique rows: {len(unique_texts)} / {len(rows)}")

if len(unique_texts) < len(rows):
    print(f"Warning: {len(rows) - len(unique_texts)} duplicate rows found")


### فحص الصفوف الفارغة أو القصيرة

لن تُسهم الصفوف الفارغة أو القصيرة جدًا في أزواج كلمات (bigrams) مفيدة. صفِّها خارجًا:


In [ ]:
short_rows = [row for row in rows if len(row["text"].split()) < 3]
print(f"Rows with fewer than 3 words: {len(short_rows)}")


تجمع دالة ملخص المتن كل هذه الفحوصات:


In [ ]:
def corpus_summary(path):
    import csv
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    texts = [row["text"] for row in rows]
    total_chars = sum(len(t) for t in texts)
    unique = len(set(texts))

    print(f"Rows: {len(rows)}")
    print(f"Unique: {unique}")
    print(f"Total chars: {total_chars:,}")
    print(f"Avg length: {total_chars / len(rows):.0f}")
    print(f"Columns: {list(rows[0].keys())}")


## جرّب بنفسك

نفّذ `corpus_summary("slm-corpus.csv")` ولاحظ:
1. كم عدد الصفوف في المتن؟
2. هل توجد أي تكرارات؟
3. هل متوسط طول النص كافٍ لبناء أزواج كلمات ذات معنى (20+ كلمة في الصف على الأقل)؟

## الخلاصات الرئيسية

- استكشف بياناتك دائمًا قبل المعالجة — افحص الأعداد والأطوال والتكرارات
- الصفوف القصيرة أو الفارغة تضيف ضجيجًا؛ صفِّها بناءً على حد أدنى لعدد الكلمات
- التكرارات تضخّم أعداد التكرارات دون إضافة أنماط جديدة
- دالة ملخص سريعة توفّر وقتًا عبر المشاريع

## تحدي الممارسة

اكتب دالة `corpus_quality(path)` تحمّل CSV وتُرجع قاموسًا بهذه المفاتيح: `"rows"`، و`"unique"`، و`"total_chars"`، و`"avg_length"`، و`"min_length"`، و`"max_length"`. استخدمها لتقييم ما إذا كان `slm-corpus.csv` مناسبًا لنمذجة أزواج الكلمات.


In [ ]:
def corpus_quality(path):
    import csv
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    texts = [row["text"] for row in rows]
    lengths = [len(t.split()) for t in texts]

    return {
        "rows": len(rows),
        "unique": len(set(texts)),
        "total_chars": sum(len(t) for t in texts),
        "avg_length": sum(lengths) / len(lengths) if lengths else 0,
        "min_length": min(lengths) if lengths else 0,
        "max_length": max(lengths) if lengths else 0,
    }


In [ ]:
# The end. Practice on your own — each cell is a minimal, runnable chunk.
